# Evaluate a fault detector — reference

Select a decision threshold on validation scores, then report a fixed test result and an always-healthy baseline. Scores are synthetic illustrative outputs, not a trained or calibrated detector.

All examples are explicitly synthetic teaching examples, not validated machinery data.
Run from a clean kernel, top to bottom. Python 3.10+ and NumPy 1.26+ are the declared target environment. See the course validation file for actual tested versions. No internet or AI API is used by these notebooks after installing dependencies.
Use the starter before consulting the reference. Record permitted AI assistance and explain your own choices.


In [ ]:
import sys
import numpy as np
print('Python',sys.version.split()[0],'NumPy',np.__version__)
val_scores=np.array([.10,.20,.35,.40,.55,.62,.70,.82,.90,.30,.65,.45])
val_labels=np.array([0,0,0,1,0,1,1,1,1,0,0,1])
test_scores=np.array([.05,.15,.28,.42,.48,.58,.68,.78,.92,.25,.72,.52])
test_labels=np.array([0,0,0,1,0,1,1,1,1,0,0,1])
print('Positive class = fault. Threshold rule: score >= threshold.')


## 1. Count outcomes

Return integer TP,FP,FN,TN for labels and predictions of shape (n,). Add precision and recall with None when the relevant denominator is zero.

In [ ]:
def counts(y,p):
    return dict(TP=int(np.sum((y==1)&(p==1))), FP=int(np.sum((y==0)&(p==1))),
                FN=int(np.sum((y==1)&(p==0))), TN=int(np.sum((y==0)&(p==0))))
def report(y,p):
    c=counts(y,p); tp,fp,fn,tn=(c[k] for k in ['TP','FP','FN','TN'])
    return dict(c,precision=tp/(tp+fp) if tp+fp else None,
                recall=tp/(tp+fn) if tp+fn else None,accuracy=(tp+tn)/len(y))


## 2. Select a threshold

Teaching cost: missed fault costs 5 units, false alarm costs 1. Evaluate thresholds .3,.4,.5,.6,.7 on validation only. Tie rule: choose the lowest threshold. This cost is illustrative, not an approved machinery policy.

In [ ]:
def choose(scores,y):
    candidates=[]
    for t in [.3,.4,.5,.6,.7]:
        c=counts(y,(scores>=t).astype(int)); cost=5*c['FN']+c['FP']
        candidates.append((cost,t)); print('validation threshold',t,'cost',cost,c)
    return min(candidates)[1]
threshold=choose(val_scores,val_labels)
print('Selected threshold',threshold)


## 3. Locked test and baseline

Report test outcomes using the selected threshold. Compare always-healthy predictions. Separately calculate accuracy and recall for 99 healthy and one missed fault.

In [ ]:
print('Test selected detector:',report(test_labels,(test_scores>=threshold).astype(int)))
print('Test always healthy:',report(test_labels,np.zeros(len(test_labels),dtype=int)))
rare=np.array([0]*99+[1]); baseline=report(rare,np.zeros(100,dtype=int))
print('Rare-fault example:',baseline)
assert baseline['accuracy']==.99 and baseline['recall']==0
assert report(np.array([1,1,0]),np.array([1,0,1]))['TP']==1


## Report
Explain class definitions, threshold selection, validation versus test roles, both baselines and cost assumptions. Explain why probabilities require calibration evidence and why synthetic scores cannot authorize machinery decisions.
